# Часть 2.2 — Upwork Job Postings (200k+)

3 эксперимента: (1) бинарная классификация `is_hourly` (часовая vs фиксированная); (2) регрессия середины часовой ставки (`hourly_high`) у часовых вакансий; (3) KMeans-сегментация вакансий.

In [1]:
from pathlib import Path

import kagglehub
import pandas as pd
from _setup import evaluate_models, make_spark
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    ClusteringEvaluator,
    RegressionEvaluator,
)
from pyspark.ml.feature import StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import functions as F

SEED = 42
spark = make_spark("hw3-upwork", partitions=8, memory="4g")
spark

## Загрузка

In [2]:
src = Path(kagglehub.dataset_download("asaniczka/all-jobs-on-upwork-200k-plus"))
df = (
    spark.read.csv(str(src / "*.csv"), header=True, inferSchema=True)
    .withColumn(
        "is_hourly_d",
        F.when(F.col("is_hourly").cast("string").isin("true", "True", "1", "TRUE"), 1.0).otherwise(0.0),
    )
    .withColumn("title_len", F.length(F.coalesce(F.col("title"), F.lit(""))))
    .withColumn("hourly_low", F.col("hourly_low").cast("double"))
    .withColumn("hourly_high", F.col("hourly_high").cast("double"))
    .withColumn("budget", F.col("budget").cast("double"))
    .select("title_len", "hourly_low", "hourly_high", "budget", "country", "is_hourly_d")
    .na.fill({"country": "unknown", "hourly_low": 0.0, "hourly_high": 0.0, "budget": 0.0, "title_len": 0})
    .filter(F.col("is_hourly_d").isNotNull())
)
df.cache()
print(f"строк: {df.count()}")
df.groupBy("is_hourly_d").count().show()

строк: 827149


+-----------+------+
|is_hourly_d| count|
+-----------+------+
|        1.0|379084|
|        0.0|448065|
+-----------+------+



## Подготовка

In [3]:
prep = Pipeline(
    stages=[
        StringIndexer(inputCol="country", outputCol="country_idx", handleInvalid="keep"),
        VectorAssembler(
            inputCols=["title_len", "hourly_low", "hourly_high", "budget", "country_idx"],
            outputCol="features_raw",
        ),
        StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False),
    ]
).fit(df)
data_cls = prep.transform(df).select("features", "is_hourly_d")

# Для регрессии hourly_high — отдельный pipeline без leakage.
prep_h = Pipeline(
    stages=[
        StringIndexer(inputCol="country", outputCol="country_idx", handleInvalid="keep"),
        VectorAssembler(inputCols=["title_len", "hourly_low", "country_idx"], outputCol="features_raw"),
        StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False),
    ]
).fit(df)
data_h = (
    prep_h.transform(df)
    .filter((F.col("is_hourly_d") == 1.0) & (F.col("hourly_high") > 0))
    .withColumnRenamed("hourly_high", "label")
    .select("features", "label")
)

train, test = data_cls.randomSplit([0.8, 0.2], seed=SEED)
train.cache()
test.cache()
tr_h, te_h = data_h.randomSplit([0.8, 0.2], seed=SEED)
tr_h.cache()
te_h.cache()
print(f"cls train: {train.count()}, test: {test.count()}")
print(f"hourly train: {tr_h.count()}, test: {te_h.count()}")

cls train: 661724, test: 165425


hourly train: 263040, test: 65417


## Эксперимент 1 — классификация is_hourly

In [4]:
cls_evaluators = {
    "ROC_AUC": BinaryClassificationEvaluator(labelCol="is_hourly_d", metricName="areaUnderROC"),
    "PR_AUC": BinaryClassificationEvaluator(labelCol="is_hourly_d", metricName="areaUnderPR"),
}
classifiers = {
    "LogReg": LogisticRegression(featuresCol="features", labelCol="is_hourly_d", maxIter=50),
    "RandomForest": RandomForestClassifier(
        featuresCol="features", labelCol="is_hourly_d", numTrees=50, seed=SEED, maxDepth=8
    ),
}
evaluate_models(classifiers, train, test, cls_evaluators)

,ROC_AUC,PR_AUC
model,,
LogReg,0.989622,0.989123
RandomForest,0.989763,0.989277


## Эксперимент 2 — регрессия hourly_high (только часовые)

In [5]:
reg_evaluators = {
    "RMSE": RegressionEvaluator(labelCol="label", metricName="rmse"),
    "R2": RegressionEvaluator(labelCol="label", metricName="r2"),
}
regressors = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="label", maxIter=50),
    "RFRegressor": RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, seed=SEED, maxDepth=8),
}
evaluate_models(regressors, tr_h, te_h, reg_evaluators)

,RMSE,R2
model,,
LinearRegression,35.519416,0.393676
RFRegressor,37.786493,0.313807


## Эксперимент 3 — KMeans на вакансиях (k=2..6)

In [6]:
sample = data_cls.sample(0.1, seed=SEED).cache()
sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
rows = []
for k in range(2, 7):
    m = KMeans(k=k, featuresCol="features", seed=SEED, maxIter=20).fit(sample)
    rows.append({
        "k": k,
        "silhouette": sil_eval.evaluate(m.transform(sample)),
        "WSSSE": m.summary.trainingCost,
    })
exp3 = pd.DataFrame(rows).set_index("k")
exp3

,silhouette,WSSSE
k,,
2,0.578733,338886.805798
3,0.623517,265708.328861
4,0.614487,213911.799001
5,0.533909,184967.282350
6,0.511044,142598.308582


## Выводы

- **Эксп. 1**: классификация `is_hourly` — ROC-AUC ≈ 0.99 у обеих моделей. `hourly_low/high` и `budget` детерминированно зависят от типа оплаты (одни нули там, где другие заполнены) — задача почти тривиальная.
- **Эксп. 2**: регрессия `hourly_high` только на часовых вакансиях по `hourly_low + country + title_len`: LR R² ≈ 0.39, RF ≈ 0.31. LR немного лучше — связь почти линейная (high ≈ low + надбавка), RF чуть переобучается.
- **Эксп. 3**: silhouette максимален при k=3 (0.62) — три устойчивых сегмента: дешёвые часовые / дорогие часовые / фиксированные.

In [7]:
spark.stop()